# Ensemble

In [1]:
#@title Librerías necesarias
import json
import random
import torch
!pip install unsloth codecarbon
import unsloth
from unsloth import FastVisionModel
from codecarbon import EmissionsTracker
import gc
import re
import os
from google.colab import drive
from PIL import Image
from tqdm import tqdm

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [15]:
#@title Montar Google Drive
drive.mount('/content/drive', force_remount=True)

BASE_PATH = "/content/drive/MyDrive/MASTER/TFMs/PROFE 2025/"

GEMMA_RESULTS_PATH = os.path.join(BASE_PATH, "gemma4_results/test/zs_gemma4_test_formatted.json")
MINISTRAL_RESULTS_PATH = os.path.join(BASE_PATH, "ministral_results/test/zs_ministral_test_formatted.json")
QWEN_RESULTS_PATH = os.path.join(BASE_PATH, "qwen35_results/test/zs_qwen35_test_formatted.json")

QUESTIONS_FILE = os.path.join(BASE_PATH, "data/test/multiple_choice_dataset.json")

Mounted at /content/drive


In [3]:
SYSTEM_PROMPT_JUEZ = """Eres un profesor experto en resolver exámenes de comprensión lectora en español.
Tu tarea es resolver las discrepancias entre alumnos que han dado respuestas distintas a una misma pregunta.
Para ello debes leer atentamente el texto y responder ÚNICAMENTE con la letra de la opción correcta.

Instrucciones:
1. Lee atentamente el texto, la pregunta, las opciones y el aviso con las respuestas de los alumnos.
2. Analiza críticamente las opciones en disputa de forma clara, lógica y concisa. Ve directo al grano: argumenta qué evidencia del texto apoya a la opción correcta y qué cosas invalidan a las demás, sin divagar.
3. Basa tu análisis EXCLUSIVAMENTE en la información proporcionada, sin utilizar ningún tipo de conocimiento externo.
4. Tras finalizar tu razonamiento interno, debes responder ÚNICAMENTE con la letra de la opción correcta.

Las ÚNICAS opciones de respuesta válidas son: {letras_validas}

IMPORTANTE: Fuera de tu bloque de pensamiento, no escribas texto adicional, justificaciones, puntos ni introducciones. Tu salida final debe ser estrictamente UNA sola letra."""

In [4]:
def load_model_unsloth(model_name, max_seq_length=2048, dtype=None, load_in_4bit=True):
    """
    Carga un modelo y su tokenizador usando Unsloth y lo prepara para inferencia.
    """
    model, tokenizer = FastVisionModel.from_pretrained(
        model_name = model_name,
        max_seq_length = max_seq_length,
        dtype = dtype,
        load_in_4bit = load_in_4bit,
    )
    FastVisionModel.for_inference(model)
    return model, tokenizer

In [5]:
def load_data(path: str) -> dict:
    print(path)
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

In [6]:
def filter_questions(data):
  """Filtra las preguntas del subset y prepara la lista de tareas a procesar."""
  tareas = []
  for exam in data['exams']:
      nivel = exam['level']
      for ex_wrapper in exam['exercises']:
          exercise = ex_wrapper['exercise']

          for q in exercise['questions']:
              q_id = q['questionId']
              tareas.append({
                  "id": q_id,
                  "nivel": nivel,
                  "contexto": exercise.get('text', ''),
                  "pregunta": q['text'],
                  "opciones": q['options']
              })
  return tareas

In [7]:
def prepare_batch(batch, text_template, output_mode):
    mensajes_batch = []
    imagenes_batch = []

    for t in batch:
        user_content = []
        imagenes_tarea = []

        # Extraemos SIEMPRE todas las opciones válidas para el System Prompt
        options_ids = [opt['optionId'] for opt in t['opciones']]
        valid_options = ", ".join(options_ids)

        if 'veredictos_previos' in t and t['veredictos_previos']:
            # Mantenemos TODOS los votos (incluyendo repeticiones) para que vea la distribución
            votos = t['veredictos_previos']
            if len(votos) > 1:
                resumen_votos = ", ".join(votos[:-1]) + f" y {votos[-1]}"
            else:
                resumen_votos = votos[0]

            aviso_juez = (
                f"\n[ATENCIÓN: Las respuestas de los alumnos han sido: {resumen_votos}. "
                f"Ten en cuenta esta tendencia, pero analiza detenidamente el texto y las opciones para dar tu respuesta definitiva.]\n"
            )
        else:
            aviso_juez = ""

        # Aplicar la plantilla del System Prompt
        system_prompt = text_template.format(letras_validas=valid_options)

        # Construcción del texto base
        base_text = f"Texto:\n{t['contexto']}\n\nPregunta: {t['pregunta']}\n"
        base_text += aviso_juez
        base_text += "\nOpciones:\n"

        user_content.append({"type": "text", "text": base_text})

        # Añadimos TODAS las opciones originales
        for opt in t['opciones']:
            letra = opt['optionId']
            texto = opt.get('text', '').strip()
            ruta_img = opt.get('image-path', '')

            if texto:
                user_content.append({"type": "text", "text": f"{letra}) {texto}\n"})

            elif ruta_img:
                user_content.append({"type": "text", "text": f"{letra}) "})

                # Carga de la imagen
                ruta_absoluta = os.path.join(BASE_PATH, ruta_img)
                img_pil = Image.open(ruta_absoluta).convert("RGB")
                imagenes_tarea.append(img_pil)

                user_content.append({"type": "image"})
                user_content.append({"type": "text", "text": "\n"})

        if output_mode == "letra":
            user_content.append({"type": "text", "text": "\nRespuesta:"})

        mensajes_batch.append([
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_content}
        ])
        imagenes_batch.append(imagenes_tarea)

    return mensajes_batch, imagenes_batch

In [8]:
def generate_response(model, tokenizer, batch_messages, batch_imagenes, max_new_tokens):
    """Ejecuta la inferencia multimodal sobre un lote y devuelve los textos generados."""

    textos_prompt = [
        tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=True, enable_thinking=True)
        for m in batch_messages
    ]

    imagenes_planas = [img for sublista in batch_imagenes for img in sublista]

    model_inputs = tokenizer(
        text=textos_prompt,
        images=imagenes_planas if len(imagenes_planas) > 0 else None,
        padding=True,
        return_tensors="pt",
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            max_length=None,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id
        )

    input_len = model_inputs.input_ids.shape[1]
    respuestas_brutas = []

    for output in outputs:
        gen_tokens = output[input_len:]
        texto = tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()
        respuestas_brutas.append(texto)

    return respuestas_brutas

In [9]:
def process_response(texto_bruto, modo_salida):
    """Extrae la letra (A-D) y la explicación, manejando el formato sin etiquetas de cierre."""
    prediccion = "N/A"
    explicacion = ""
    error_formato = False

    # 1. Extraer el pensamiento (todo lo que hay después de 'thought\n' si existe)
    pensamiento_interno = ""
    if texto_bruto.startswith("thought\n") or "<|channel>thought" in texto_bruto:
        # Quitamos la palabra inicial para limpiar la explicación
        pensamiento_interno = re.sub(r'^(?:<\|channel>)?thought\n?', '', texto_bruto).strip()

    if modo_salida == "letra":
        # LA MAGIA ESTÁ AQUÍ: Buscamos la ÚLTIMA letra A, B, C o D que aparezca al final del texto.
        # [A-D] busca la letra, [^a-zA-Z]*$ asegura que no haya otras letras después de ella hasta el final.
        match = re.search(r'([A-D])[^a-zA-Z]*$', texto_bruto.upper())

        if match:
            prediccion = match.group(1)
        else:
            error_formato = True

        if pensamiento_interno:
            # Quitamos la letra final pegada al texto para que la explicación quede limpia
            if match:
                explicacion = f"{pensamiento_interno[:-1].strip()}"
            else:
                explicacion = f"{pensamiento_interno}"

    return prediccion, explicacion, error_formato

In [10]:
def split_tasks_by_modality(tareas):
    """Separa las tareas en dos grupos: de solo texto y con imágenes."""
    tareas_texto = []
    tareas_imagen = []

    for t in tareas:
        tiene_imagen = False
        if isinstance(t.get('opciones'), list):
            tiene_imagen = any(opt.get('image-path', '') != '' for opt in t['opciones'])

        if tiene_imagen:
            tareas_imagen.append(t)
        else:
            tareas_texto.append(t)

    return tareas_texto, tareas_imagen

In [11]:
def run_inference_ensemble(
    model,
    tokenizer,
    system_prompt_text_juez,
    dataset_path,
    prediction_paths,
    modo_salida="json",
    max_new_tokens=2048,
    batch_size_texto=1,
    output_file="resultados_ensemble.jsonl"
):
    """Ejecuta inferencia (Juez) SOLO en las preguntas donde el ensemble no tiene unanimidad."""

    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # 1. Cargar el dataset original
    dataset_bruto = load_data(dataset_path)
    todas_las_tareas = filter_questions(dataset_bruto)

    # 2. Cargar las predicciones de los modelos (¡Ahora son diccionarios directos!)
    print(f"Cargando diccionarios de predicciones de {len(prediction_paths)} modelos...")

    # Cada elemento en mapas_respuestas será un dict: {"A1_...": "A", "A2_...": "C"}
    mapas_respuestas = [load_data(path) for path in prediction_paths]

    directorio_salida = os.path.dirname(output_file)
    if directorio_salida: # Comprueba que la ruta no esté vacía
        os.makedirs(directorio_salida, exist_ok=True)

    # Si el archivo ya existía de una prueba anterior, lo borramos para empezar limpio
    if os.path.exists(output_file):
        os.remove(output_file)

    tareas_acuerdo = []
    tareas_desacuerdo = []

    # 3. Lógica de separación: Acuerdo vs Desacuerdo
    for tarea in todas_las_tareas:
        q_id = tarea["id"]

        # Extraer qué respondió cada modelo para este ID buscándolo en los diccionarios
        # Usamos .get(q_id) que devolverá None si el modelo no respondió a esa pregunta
        respuestas_para_este_id = [mapa.get(q_id) for mapa in mapas_respuestas]

        # Comprobar si tenemos respuestas de todos y si todas son idénticas
        if len(respuestas_para_este_id) == len(prediction_paths) and None not in respuestas_para_este_id and len(set(respuestas_para_este_id)) == 1:
            # UNANIMIDAD
            tareas_acuerdo.append({
                "questionId": q_id,
                "nivel": tarea.get("nivel", "Desconocido"),
                "pregunta": tarea["pregunta"],
                "prediccion_modelo": respuestas_para_este_id[0],
                "respuesta_completa": "",
                "explicacion": "",
                "error_procesamiento_json": None,
                "metodo": "unanimidad"
            })
        else:
            # DESACUERDO
            tarea_juez = tarea.copy()
            # Inyectamos los veredictos previos para que prepare_batch los procese
            # Filtramos los None por si algún modelo falló en predecir esa pregunta específica
            tarea_juez["veredictos_previos"] = [res for res in respuestas_para_este_id if res is not None]
            tareas_desacuerdo.append(tarea_juez)

    # 4. Escribir los acuerdos directamente en el archivo final
    print(f"\nSe encontraron {len(tareas_acuerdo)} preguntas con UNANIMIDAD. Guardando directamente...")
    with open(output_file, 'a', encoding='utf-8') as f:
        for resultado in tareas_acuerdo:
            f.write(json.dumps(resultado, ensure_ascii=False) + '\n')

    if not tareas_desacuerdo:
        print("¡Todos los modelos coincidieron en todas las preguntas! Inferencia finalizada.")
        return

    # 5. Procesar los desacuerdos usando tu lógica original
    print(f"\nSe encontraron {len(tareas_desacuerdo)} preguntas con DESACUERDO. Iniciando Juez LLM...")
    tareas_texto, tareas_imagen = split_tasks_by_modality(tareas_desacuerdo)

    def procesar_grupo(grupo_tareas, b_size, descripcion):
        for i in tqdm(range(0, len(grupo_tareas), b_size), desc=descripcion):
            batch = grupo_tareas[i : i + b_size]

            mensajes, imagenes = prepare_batch(batch, system_prompt_text_juez, modo_salida)
            textos_generados = generate_response(model, tokenizer, mensajes, imagenes, max_new_tokens)

            batch_results = []
            for j, texto_bruto in enumerate(textos_generados):
                prediccion, explicacion, errors = process_response(texto_bruto, modo_salida)
                tarea_actual = batch[j]

                batch_results.append({
                    "questionId": tarea_actual["id"],
                    "nivel": tarea_actual.get("nivel", ""),
                    "pregunta": tarea_actual["pregunta"],
                    "prediccion_modelo": prediccion,
                    "respuesta_completa": texto_bruto,
                    "explicacion": explicacion,
                    "error_procesamiento_json": errors,
                    "metodo": "llm_juez"
                })

            with open(output_file, 'a', encoding='utf-8') as f:
                for resultado in batch_results:
                    f.write(json.dumps(resultado, ensure_ascii=False) + '\n')

    if tareas_texto:
        print(f"\n--- Procesando {len(tareas_texto)} desacuerdos de TEXTO (Batch: {batch_size_texto}) ---")
        procesar_grupo(tareas_texto, batch_size_texto, "Progreso Texto Juez")

    if tareas_imagen:
        print(f"\n--- Procesando {len(tareas_imagen)} desacuerdos MULTIMODALES (Batch: 1) ---")
        procesar_grupo(tareas_imagen, 1, "Progreso Imágenes Juez")

    print(f"\nResultados completos guardados en: {output_file}")

## Gemma4


In [12]:
model, tokenizer = load_model_unsloth("unsloth/gemma-4-E4B-it-unsloth-bnb-4bit", max_seq_length=2048, load_in_4bit=True)

==((====))==  Unsloth 2026.5.2: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/2130 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/203 [00:00<?, ?B/s]

processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

In [13]:
path = os.path.join(BASE_PATH, 'ensemble_results')
ensemble_results_path = os.path.join(path, "ensemble_test.json")


In [16]:
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Configure emissions tracker
tracker = EmissionsTracker(
    project_name="ensemble_test",
    output_dir=path,
    output_file="ensemble_test_emissions.csv",
    log_level="warning"
)

RUTAS_PREDICCIONES = [GEMMA_RESULTS_PATH, MINISTRAL_RESULTS_PATH, QWEN_RESULTS_PATH]

tracker.start()
try:
  run_inference_ensemble(
      model=model,
      tokenizer=tokenizer,
      system_prompt_text_juez=SYSTEM_PROMPT_JUEZ, # Asegúrate de que sea el prompt diseñado para el juez
      dataset_path=QUESTIONS_FILE,
      prediction_paths=RUTAS_PREDICCIONES,
      modo_salida="letra",       # Mantenemos "letra" según tu script anterior
      max_new_tokens=2048,       # SUBIDO A 2048: Recomendado si usas enable_thinking=True (Chain of Thought)
      batch_size_texto=8,        # Mantenemos en 1
      output_file=ensemble_results_path
  )
finally:
  emisiones = tracker.stop()
  print(f"Inferencia completada.")
  print(f"Emisiones estimadas: {emisiones:.4f} kg de CO2eq")

[codecarbon WARNING @ 12:13:22] We saw that you have a Intel(R) Xeon(R) CPU @ 2.20GHz but we don't know it. Please contact us.
[codecarbon WARNING @ 12:13:22] No CPU tracking mode found. Falling back on estimation based on TDP for CPU. 
 Linux OS detected: Please ensure RAPL files exist, and are readable, at /sys/class/powercap/intel-rapl/subsystem to measure CPU

[codecarbon WARNING @ 12:13:22] No CPU tracking mode found. Falling back on CPU constant mode.


/content/drive/MyDrive/MASTER/TFMs/PROFE 2025/data/test/multiple_choice_dataset.json
Cargando diccionarios de predicciones de 3 modelos...
/content/drive/MyDrive/MASTER/TFMs/PROFE 2025/gemma4_results/test/zs_gemma4_test_formatted.json
/content/drive/MyDrive/MASTER/TFMs/PROFE 2025/ministral_results/test/zs_ministral_test_formatted.json
/content/drive/MyDrive/MASTER/TFMs/PROFE 2025/qwen35_results/test/zs_qwen35_test_formatted.json

Se encontraron 1464 preguntas con UNANIMIDAD. Guardando directamente...

Se encontraron 366 preguntas con DESACUERDO. Iniciando Juez LLM...

--- Procesando 349 desacuerdos de TEXTO (Batch: 8) ---


Progreso Texto Juez: 100%|██████████| 44/44 [2:36:00<00:00, 212.73s/it]



--- Procesando 17 desacuerdos MULTIMODALES (Batch: 1) ---


Progreso Imágenes Juez: 100%|██████████| 17/17 [31:42<00:00, 111.94s/it]


Resultados completos guardados en: /content/drive/MyDrive/MASTER/TFMs/PROFE 2025/ensemble_results/ensemble_test.json
Inferencia completada.
Emisiones estimadas: 0.2131 kg de CO2eq


In [18]:
def procesar_json(archivo_entrada, archivo_salida):
    resultado = {}

    # Abrimos el archivo de entrada en modo lectura
    with open(archivo_entrada, 'r', encoding='utf-8') as f_entrada:
        for linea in f_entrada:
            # Limpiamos los espacios en blanco y saltos de línea
            linea = linea.strip()
            if linea: # Verificamos que la línea no esté vacía
                try:
                    # Convertimos la cadena de texto a un diccionario de Python
                    datos = json.loads(linea)

                    # Extraemos los valores deseados
                    q_id = datos.get("questionId")
                    prediccion = datos.get("prediccion_modelo")

                    # Si ambas claves existen, las agregamos al diccionario final
                    if q_id and prediccion is not None:
                        resultado[q_id] = prediccion

                except json.JSONDecodeError:
                    print(f"Error al procesar la línea: {linea}")

    # Guardamos el diccionario resultante en un nuevo archivo JSON
    with open(archivo_salida, 'w', encoding='utf-8') as f_salida:
        # indent=4 le da el formato estructurado y legible que buscas
        json.dump(resultado, f_salida, indent=4, ensure_ascii=False)
ensemble_processed_path = os.path.join(path, "ensemble_test_formatted.json")
procesar_json(ensemble_results_path, ensemble_processed_path)
print(f"Procesamiento completado. El archivo ha sido guardado como '{ensemble_processed_path}'.")

Procesamiento completado. El archivo ha sido guardado como '/content/drive/MyDrive/MASTER/TFMs/PROFE 2025/ensemble_results/ensemble_test_formatted.json'.
